In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
from PIL import Image

from rtnls_inference.ensembles import HeatmapRegressionEnsemble

# Fovea localisation

This example loads the fovea model from Hugging Face and returns one `(x, y)` point per image. The release predicts a heatmap, so we use `HeatmapRegressionEnsemble`; it handles conversion to coordinates for us.

Run the notebook from this directory. The first model load downloads the release; subsequent loads can reuse the local Hugging Face cache.

In [ ]:
# Sample raw fundus photographs, relative to this notebook's directory.
INPUT_DIR = Path("../samples/fundus/original")

# Use "preprocessed" for images that already have the canonical fundus crop.
INPUT_SPACE = "raw"

# Prefer GPU 1 when available, then GPU 0, otherwise use CPU.
DEVICE = torch.device("cuda:1" if torch.cuda.device_count() > 1 else "cuda:0" if torch.cuda.is_available() else "cpu")

# Number of images processed together. Reduce this if GPU memory is limited.
BATCH_SIZE = 2

# Fovea heatmap model hosted on Hugging Face.
FOVEA_MODEL = "Eyened/vascx:fovea/fovea_may26.pt"

In [ ]:
image_paths = sorted(
    path for path in INPUT_DIR.iterdir()
    if path.suffix.lower() in {".jpg", ".jpeg", ".png", ".tif", ".tiff"}
)
print(f"Found {len(image_paths)} images; using {DEVICE}.")

ensemble = HeatmapRegressionEnsemble.from_huggingface(FOVEA_MODEL).to(DEVICE).eval()

## 1. Single-input inference

Preprocess → make a batch → predict → postprocess. The example starts with a loaded RGB uint8 NumPy image. A file path or uint8 CHW tensor works in the same `image` field.

`result["output"]` is an array of `(x, y)` points. This model returns one point, so its shape is `(1, 2)`. Coordinates are pixels in `result["preprocessed_image"]`, normally 1024 × 1024, with the origin at the top left. They are not coordinates in the original photograph. Always use the same batch for prediction and postprocessing so the point is scaled correctly.

In [ ]:
# Choose a sample with the macular region visible.
image_path = INPUT_DIR / "DRIVE_22.png"
with Image.open(image_path) as image:
    raw_image = np.array(image.convert("RGB"))

# 1. Preprocess the loaded RGB image. No ID is needed for an in-memory result.
item = ensemble.preprocess_item({"image": raw_image}, input_space=INPUT_SPACE)

# 2. Make a batch of size 1.
batch = ensemble.make_batch([item])

# 3. Run the ensemble.
predictions = ensemble.predict_batch(batch)

# 4. Postprocess using the same batch's input context.
result = ensemble.postprocess_batch(predictions, batch)[0]

x, y = result["output"][0]
print(f"Fovea: x={x:.1f}, y={y:.1f} pixels in the preprocessed image")

In [ ]:
plt.figure(figsize=(6, 6))
plt.imshow(result["preprocessed_image"])
plt.scatter([x], [y], marker="+", s=220, linewidths=2, color="cyan", label="Predicted fovea")
plt.title(image_path.name)
plt.legend()
plt.axis("off")
plt.show()

## 2. Batch inference

Create the input records and a dataloader, then call `predict_dataloader` to return a compact table. It uses the same preprocessing and postprocessing as the single-image example. No destination is supplied, so no prediction files are written.

The grid shows each preprocessed image with a cyan cross marking the predicted fovea.

Path inputs get IDs from their filename stems, which must be unique. IDs are optional for already-loaded images when returning results in memory.

Input records are ordinary dictionaries. Creating the dataloader validates them once; image files are loaded when the loader is iterated. Predictions and geometry are dictionaries too.


In [ ]:
# 1. Create records. Each filename stem becomes the image's ID.
data = [{"image": path} for path in image_paths]

# 2. Create the inference dataloader.
loader = ensemble.make_inference_dataloader(
    data,
    input_space=INPUT_SPACE,
    batch_size=BATCH_SIZE,
    num_workers=0,
)

In [ ]:
# 3. Return a compact prediction table in memory. No destination means no files.
results = ensemble.predict_dataloader(loader)
results


In [ ]:
# Arrange all sample images in a grid, with three images per row.
ncols = 3
nrows = (len(results) + ncols - 1) // ncols
fig, axes = plt.subplots(nrows, ncols, figsize=(5 * ncols, 5 * nrows), squeeze=False)
for ax in axes.flat:
    ax.axis("off")
for ax, path, result in zip(axes.flat, image_paths, results.to_dict("records")):
    # Prepare the canonical image for display; the results table holds only coordinates.
    image = ensemble.preprocess_item({"image": path}, input_space=INPUT_SPACE)["preprocessed_image"]
    ax.imshow(image)
    ax.scatter([result["x0"]], [result["y0"]], marker="+", s=220, linewidths=2, color="cyan")
    ax.set_title(result["id"])
fig.tight_layout()
plt.show()
